# Fig 6D and additional locus exports — Robust multi-CS AD GWAS loci (data processing)

The original main examples use loci with matching RSS/rRSS CS counts. The final Extended Data Figure 3 catalog cell below instead exports every final rRSS multi-CS analysis, without an RSS-agreement filter. Its publication plot shows only PIPs and credible sets.

**Outputs:**
- `figure_data/Fig6D.AD_GWAS_Multi_CS_Loci.rds` — named list of loci (`load_locus()` in `R/utils.R`; PLCG2, ABCA7, …)
- `figure_data/FigED3.AD_GWAS_Multi_CS_All_Studies.rds` — per block: SuSiE-rRSS PIPs for every study with a multi-CS result there, plus the AD xQTL evidence

Each locus / block also carries `tm`: the 2026 AD GWAS colocalization (TransMap, p < 1e-4) variables inside the block with their VCP and whether the colocalized outcomes are cis, trans or both (`tidy_transmap()` / `locus_transmap()` in `R/utils.R`).

In [1]:
source("../../R/utils.R")
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)   # getSumStatsDf()
library(readxl)
filter <- dplyr::filter; select <- dplyr::select

ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”


In [2]:
### Load AD GWAS explorer data
AD_loci <- suppressMessages(
  suppressWarnings(
    read_excel(
      "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/unified_AD_loci_xQTL_summary.xlsx",
      skip = 2
    )
  )
)

cs_counts_df <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export//method_compare.cs_counts.rds")

multi_cs <- cs_counts_df %>% filter(block_id != apoe_block) %>% filter(n_cs_ebmix == n_cs_susie & n_cs_ebmix > 1) %>% mutate(index = row_number())

## 2026 AD GWAS colocalization (TransMap)
One row per colocalized variable, with its VCP and whether the colocalization's outcomes are cis, trans or both.

In [3]:
transmap <- read_tsv("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/transmap_AD_target_results_pv1e4.tsv")

transmap_long <- tidy_transmap(transmap)
transmap_long %>% count(focal_outcome, context)

Rows: 228 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): focal_outcome, colocalized_outcomes, cos_id, top_variable, colocali...
dbl (5): purity, top_variable_vcp, cos_npc, min_npc_outcome, n_variables

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


focal_outcome,context,n
<chr>,<chr>,<int>
AD_Bellenguez_2026,both,8210
AD_Bellenguez_2026,cis,2
AD_Bellenguez_2026,trans,1333


## Fig 6E: one study per locus

In [4]:
## rows of multi_cs shown (block / study as printed in the original notebook)
show_idx <- c(1,    # chr10_10500888_12817813    / AD_Bellenguez_noproxy_2026
              3,    # chr11_119829118_122612025  / AD_Bellenguez_main_2026
              9,    # chr11_56858541_63818332    / AD_Bellenguez_main_2026
              10,   # chr14_92293029_94303139    / AD_Bellenguez_2022
              19,   # chr16_81293081_82644764    / AD_Bellenguez_main_2026      (PLCG2)
              22,   # chr19_53788025_55215821    / AD_Bellenguez_2022
              26,   # chr19_90949_1507536        / AD_Bellenguez_2022           (ABCA7)
              36,   # chr2_125689597_127728648   / AD_Bellenguez_2022
              38,   # chr3_154360351_156292136   / AD_Bellenguez_main_2026
              40,   # chr6_40377803_42070711     / AD_Bellenguez_2022
              42,   # chr8_143034200_145075767   / AD_Bellenguez_main_2026
              45,   # chr8_26225312_29469590     / AD_Bellenguez_2022
              48,   # chr8_26225312_29469590     / AD_Wightman_Excluding23andMe_2021
              49,   # chr8_26225312_29469590     / AD_Wightman_ExcludingUKBand23andME_2021
              51,   # chr17_60570445_65149278    / AD_Bellenguez_main_2026
              53)   # chr21_24215031_26405838    / AD_Bellenguez_noproxy_2026

loci <- map(show_idx, ~ load_locus(multi_cs$block_id[.x], multi_cs$study[.x], AD_loci))
names(loci) <- map_chr(loci, ~ paste(.x$sid, .x$bid, sep = "."))
names(loci)

## 2026 AD GWAS colocalization rows inside each block
loci <- map(loci, function(loc) { loc$tm <- locus_transmap(transmap_long, loc$reg); loc })
map_int(loci, ~ nrow(.x$tm))

save_data(loci, "Fig6D.AD_GWAS_Multi_CS_Loci")

[1] "AD_Bellenguez_noproxy_2026.chr10_10500888_12817813"            
 [2] "AD_Bellenguez_main_2026.chr11_119829118_122612025"             
 [3] "AD_Bellenguez_main_2026.chr11_56858541_63818332"               
 [4] "AD_Bellenguez_2022.chr14_92293029_94303139"                    
 [5] "AD_Bellenguez_main_2026.chr16_81293081_82644764"               
 [6] "AD_Bellenguez_2022.chr19_53788025_55215821"                    
 [7] "AD_Bellenguez_2022.chr19_90949_1507536"                        
 [8] "AD_Bellenguez_2022.chr2_125689597_127728648"                   
 [9] "AD_Bellenguez_main_2026.chr3_154360351_156292136"              
[10] "AD_Bellenguez_2022.chr6_40377803_42070711"                     
[11] "AD_Bellenguez_main_2026.chr8_143034200_145075767"              
[12] "AD_Bellenguez_2022.chr8_26225312_29469590"                     
[13] "AD_Wightman_Excluding23andMe_2021.chr8_26225312_29469590"      
[14] "AD_Wightman_ExcludingUKBand23andME_2021.chr8_26225312_29469590"
[15] "AD_Bellenguez_main_2026.chr17_60570445_65149278"               
[16] "AD_Bellenguez_noproxy_2026.chr21_24215031_26405838"

AD_Bellenguez_noproxy_2026.chr10_10500888_12817813 
                                                            10 
             AD_Bellenguez_main_2026.chr11_119829118_122612025 
                                                             0 
               AD_Bellenguez_main_2026.chr11_56858541_63818332 
                                                           153 
                    AD_Bellenguez_2022.chr14_92293029_94303139 
                                                            96 
               AD_Bellenguez_main_2026.chr16_81293081_82644764 
                                                             6 
                    AD_Bellenguez_2022.chr19_53788025_55215821 
                                                            28 
                        AD_Bellenguez_2022.chr19_90949_1507536 
                                                             3 
                   AD_Bellenguez_2022.chr2_125689597_127728648 
                                                            46 
              AD_Bellenguez_main_2026.chr3_154360351_156292136 
                                                           154 
                     AD_Bellenguez_2022.chr6_40377803_42070711 
                                                            28 
              AD_Bellenguez_main_2026.chr8_143034200_145075767 
                                                            15 
                     AD_Bellenguez_2022.chr8_26225312_29469590 
                                                            30 
      AD_Wightman_Excluding23andMe_2021.chr8_26225312_29469590 
                                                            30 
AD_Wightman_ExcludingUKBand23andME_2021.chr8_26225312_29469590 
                                                            30 
               AD_Bellenguez_main_2026.chr17_60570445_65149278 
                                                            15 
            AD_Bellenguez_noproxy_2026.chr21_24215031_26405838 
                                                            65

## Additional locus exports: every multi-CS block, all studies

In [5]:
## SuSiE-rRSS PIPs of every study with a multi-CS result in block `bid`,
## plus the AD xQTL evidence for the block (plotted by plot_region_allstudies())
load_region_allstudies <- function(bid, multi_cs, AD_loci,
    root      = "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10",
    ebmix_sub = "AD_GWAS_rss_ebmix_r4_L10") {

  studies <- multi_cs %>% filter(block_id == bid) %>% distinct(study) %>% pull(study)
  m   <- str_match(bid, "^chr([0-9XY]+)_([0-9]+)_([0-9]+)$")
  reg <- list(chr = m[, 2], start = as.numeric(m[, 3]), end = as.numeric(m[, 4]))

  get_fit <- function(f) attributes(readRDS(f))$elementMetadata$susieFit[[1]]
  pip_list <- setNames(lapply(studies, function(sid) {
    f <- file.path(root, ebmix_sub, "output/fine_mapping", paste0(sid, ".", bid, ".gwas_finemap.rds"))
    if (!file.exists(f)) return(NULL)
    locus_tidy_fit(get_fit(f), sid)
  }), studies)
  pip_list <- Filter(Negate(is.null), pip_list)

  AD_subset <- AD_loci %>% filter(Chr == as.integer(reg$chr), Pos >= reg$start, Pos <= reg$end)

  list(pip_list = pip_list, ev = locus_evidence(AD_subset), reg = reg, bid = bid)
}

unique_regions <- multi_cs %>% pull(block_id) %>% unique()

regions <- map(setNames(unique_regions, unique_regions), function(bid) {
  tryCatch(load_region_allstudies(bid, multi_cs, AD_loci),
           error = function(e) { message("FAILED: ", bid, " : ", conditionMessage(e)); NULL })
})
regions <- compact(regions)
length(regions)

regions <- map(regions, function(r) { r$tm <- locus_transmap(transmap_long, r$reg); r })

save_data(regions, "FigED3.AD_GWAS_Multi_CS_All_Studies")

[1] 18

## Complete regional catalog roster (Extended Data Figure 3)

The final catalog includes every study–region analysis with multiple reported rRSS credible sets. Agreement with the baseline RSS CS count is not required. This cell supersedes the exploratory selection above for the all-study catalog.

In [ ]:
# Current catalog roster. Include every final multi-CS fit, regardless of RSS agreement.
# Run the marginal/joint preparation notebook first to refresh the exported-CS roster.
catalog_roster <- load_data("FigS12B.AD_GWAS_CS_Top_Variant_Marginal_P") %>%
  filter(as.character(region)=="Multiple credible sets") %>% distinct(block_id,study)
stopifnot(nrow(catalog_roster)>0L, !anyDuplicated(paste(catalog_roster$block_id,catalog_roster$study)))
# The existing loader reads full variant PIPs from each saved adjusted fit.
catalog_regions <- map(setNames(unique(catalog_roster$block_id),unique(catalog_roster$block_id)),function(bid)
  load_region_allstudies(bid,catalog_roster,AD_loci))
expected_keys <- paste(catalog_roster$block_id,catalog_roster$study)
loaded_keys <- unlist(imap(catalog_regions,function(r,bid)paste(bid,names(r$pip_list))))
missing_keys <- setdiff(expected_keys,loaded_keys)
if(length(missing_keys)) warning("Missing full PIP tracks: ",paste(missing_keys,collapse="; "))
# Missing tracks are left for the publication export to mark; no fit is substituted.
save_data(catalog_regions,"FigED3.AD_GWAS_Multi_CS_All_Studies")
